In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from easydynamics.job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data
from easydynamics.analysis import Analysis

from easydynamics.sample import BrownianTranslationalDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import Lorentzian
from easydynamics.sample import DeltaFunction
from easydynamics.sample import Polynomial

from easydynamics.sample import Gaussian

from easydynamics.resolution import ResolutionHandler

from easyscience import Parameter

import scipp as sc

import plopp as pp

%matplotlib widget

In [ ]:
# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,201)

diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(Q),len(E)))

scale=0.7 #arbitrary scale factor for diffusion model


T=3

model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
HWHM=model.calculate_width(Q)

QQISF=model.calculate_QISF(Q)
EISF=model.calculate_EISF(Q)

resolution=Gaussian(name="Resolution", area=1,width=0.1)

resolution_handler=ResolutionHandler()

sample_model=[]
for i in range(len(Q)):
    sample_model.append(SampleModel(name=f"SampleModel_{i}"))

    sample_model[i].add_component(DeltaFunction(area=scale*EISF[i]+0.23, name="Elastic"))
    sample_model[i].add_component(Lorentzian(area=scale*QQISF[i], name="QuasiElastic", width=HWHM[i]) )

    convoluted_signal[i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.05+0.01*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Q','energy'],values=convoluted_signal,variances=0.01*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp})


# pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])



In [ ]:
pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])


In [ ]:


diffusion_job= Job(name='BrownianDiffusion')


exp=Experiment()
data=Data()
data.append(diffusion_data)

exp.set_data(data)

diffusion_job.set_experiment(exp)
diffusion_job.generate_empty_analysis_array()


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[0.5]))
diffusion_job.set_background_model(bg)
diffusion_job.set_background_model_for_all_analyses()

resolution=SampleModel()
resolution.add_component(Gaussian(name="Resolution", area=1,width=0.1))
diffusion_job.set_resolution_model(resolution)
diffusion_job.set_resolution_model_for_all_analyses()




diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3,scale=1.0)
diffusion_job.set_diffusion_model(diffusion_model)
# diffusion_job.set_theory_for_all_analyses(diffusion_model)



# delta_model=SampleModel(name="DeltaModel")
# delta_model.add_component(DeltaFunction(name="Delta",area=1.0))
# diffusion_job.set_theory_for_all_analyses(delta_model)

# delta_model=SampleModel(name="DeltaModel")
delta_model=DeltaFunction(name="Delta",area=0.2)
diffusion_job.set_theory_for_all_analyses(delta_model)


In [ ]:
diffusion_job._analysis

In [ ]:
diffusion_job.analysis[5].get_parameters()

In [ ]:
diffusion_job.analysis[5].get_fit_parameters()

In [ ]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

In [ ]:
diffusion_job._analysis[0]._theory

diffusion_job._analysis[0]._experiment._data.data.values
diffusion_job._analysis[0]._experiment._data.data.coords['energy'].values

In [ ]:
result=diffusion_job.fit_simultaneous()


In [ ]:

result

In [ ]:
diffusion_job.plot_data_and_model(intensity_min=0.0, intensity_max=2,
                            energy_min=-5, energy_max=5)

In [ ]:
diffusion_job.analysis[5].get_parameters()

In [ ]:
diffusion_job.analysis[5].get_fit_parameters()

In [ ]:
pars=diffusion_job.analysis[5].get_parameters()
pars[2].dependency_expression

In [ ]:
diffusion_job.analysis[0].get_fit_parameters()

In [ ]:
diffusion_job_sequential= Job(name='BrownianDiffusionSequential')
diffusion_job_sequential.set_experiment(exp)
# diffusion_job_sequential.generate_empty_analysis_array()
diffusion_job_sequential.set_background_model(bg)
# diffusion_job_sequential.set_background_model_for_all_analyses()
diffusion_job_sequential.set_resolution_model(resolution)
# diffusion_job_sequential.set_resolution_model_for_all_analyses()

sequential_model=SampleModel(name="SequentialModel")
sequential_model.add_component(Lorentzian(name="QuasiElastic", area=1.0, width=0.5))
sequential_model.add_component(DeltaFunction(name="Elastic", area=0.2))
diffusion_job_sequential.set_theory(sequential_model)
diffusion_job_sequential.generate_analysis_for_cuts()

diffusion_job_sequential.plot_data_and_model(intensity_min=0.0, intensity_max=4,
                            energy_min=-5, energy_max=5)

In [ ]:
diffusion_job_sequential.analysis[5].get_parameters()

In [ ]:

diffusion_job_sequential.fit()

In [ ]:
diffusion_job_sequential.plot_data_and_model(intensity_min=0.0, intensity_max=2,
                            energy_min=-5, energy_max=5)

In [ ]:
diffusion_job_sequential.plot_fit_parameters("QuasiElastic width")

In [ ]:
diffusion_job_sequential._diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3,scale=1.0)
fit_result=diffusion_job_sequential.fit_diffusion_width("QuasiElastic width")

In [ ]:
diffusion_job_sequential.plot_diffusion_fit_result("QuasiElastic width")

In [ ]:
diffusion_job_sequential._diffusion_model.get_parameters()

In [ ]:
diffusion_job_sequential.analysis[5].get_parameters()